In [1]:
from llm_from_scratch.libs.gpt import GPT

In [2]:
model = GPT(
    vocab_size=50257,
    n_layers=12,
    d_in=768,
    context_length=1024,
    dropout=0.1,
    n_heads=12,
    qkv_bias=True,
)

sum(p.numel() for p in model.parameters())
# ~124M (may differ slightly from 124M if your block differs from OpenAI’s)

124439808

In [10]:
from llm_from_scratch.libs.tokenizer import Tokenizer
import torch

In [35]:
ctx_len = 5

model = GPT(
    vocab_size=50257,
    n_layers=12,
    d_in=768,
    context_length=ctx_len,
    dropout=0.1,
    n_heads=12,
    qkv_bias=True,
)

sum(p.numel() for p in model.parameters())
# ~124M (may differ slightly from 124M if your block differs from OpenAI’s)

# Generate Text
def generate_text(model, input, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        chopped_input = input[:,-context_size:]
        print()
        print(chopped_input)
        
        with torch.no_grad():
            logits = model(chopped_input)
        logits = logits[:,-1,:]

        probs = torch.softmax(logits, dim=-1)
        token_next = torch.argmax(probs, dim=-1, keepdim=True)
        input = torch.cat((input,token_next), dim=-1)

        print(input)
    return input


tokenizer = Tokenizer("gpt2")
input_tensor = torch.tensor(tokenizer.encode("Hi there, how are you?")).unsqueeze(0)
output = generate_text(model, input,4,ctx_len)


tensor([[ 11, 703, 389, 345,  30]])
tensor([[17250,   612,    11,   703,   389,   345,    30,    30]])

tensor([[703, 389, 345,  30,  30]])
tensor([[17250,   612,    11,   703,   389,   345,    30,    30,    30]])

tensor([[389, 345,  30,  30,  30]])
tensor([[17250,   612,    11,   703,   389,   345,    30,    30,    30,    30]])

tensor([[345,  30,  30,  30,  30]])
tensor([[17250,   612,    11,   703,   389,   345,    30,    30,    30,    30,
            30]])


In [44]:
tokenizer.decode(output.squeeze(0).tolist())

'Hi there, how are you?????'